# 03: Smoke Test + Baselines + ORA

Workhorse notebook. Mounts `gvla-weights` and `gvla-data`, runs the smoke test,
then evaluates the three agents (ReAct + Mistral, single-shot LLaVA, ORA + LLaVA)
across ScienceQA, the synthetic sample, and Mind2Web.

**Settings:** Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`.

**Datasets to attach (Add data → Your datasets):**
- `gvla-weights`
- `gvla-data`

Eval is checkpointed every task, so a session timeout is recoverable — re-run with
`resume=True` and it picks up where it left off.

In [ ]:
# Mount paths + offline mode so transformers never tries to re-download.
import os
WEIGHTS = '/kaggle/input/gvla-weights/hf_cache'
DATA    = '/kaggle/input/gvla-data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
assert os.path.isdir(WEIGHTS), 'mount the gvla-weights Kaggle Dataset first'
assert os.path.isdir(DATA),    'mount the gvla-data Kaggle Dataset first'

In [ ]:
# Repo + GPU stack. Clone on first run, pull on re-run to get latest fixes.
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
!if [ ! -d /kaggle/working/grounded_vla ]; then git clone {REPO_URL} /kaggle/working/grounded_vla; else git -C /kaggle/working/grounded_vla pull; fi
%cd /kaggle/working/grounded_vla
!pip install -q -e .[gpu]
!nvidia-smi

In [ ]:
# Smoke test: pure-mock pipeline. Should print 'smoke ok' in <5 seconds.
!grounded-vla smoke

In [ ]:
# Build the three agents.
from grounded_vla.agents import ORAAgent, ReActAgent, SingleShotVLMAgent
from grounded_vla.backends import make_backend
from grounded_vla.backends.base import GenerationConfig

WEIGHTS = '/kaggle/input/gvla-weights/hf_cache'

def build_react():
    backend = make_backend({
        'kind': 'mistral',
        'model_id': f'{WEIGHTS}/Mistral-7B-Instruct-v0.2',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ReActAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.1))

def build_single_shot():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return SingleShotVLMAgent(backend, GenerationConfig(max_new_tokens=192, temperature=0.0))

def build_ora():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ORAAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.1))

In [ ]:
# Build the three datasets pointing at the mounted gvla-data.
from grounded_vla.data import make_dataset
DATA = '/kaggle/input/gvla-data'

def scienceqa(limit=50):
    return make_dataset({
        'kind': 'scienceqa',
        'jsonl_path': f'{DATA}/scienceqa/test.jsonl',
        'images_dir': f'{DATA}/scienceqa',
        'limit': limit,
    })

def _mind2web_jsonl():
    # Auto-discover whichever split notebook 02 produced (test_task,
    # test_website, train fallback, ...).
    import glob
    candidates = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
    if not candidates:
        raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
    return candidates[0]

def mind2web(limit=30):
    return make_dataset({
        'kind': 'mind2web',
        'jsonl_path': _mind2web_jsonl(),
        'images_dir': f'{DATA}/mind2web/images',
        'limit': limit,
    })

def synthetic():
    return make_dataset({
        'kind': 'jsonl',
        'path': f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
        'source': 'synthetic',
    })

In [ ]:
# Generic eval loop with checkpointing. Resumable across sessions.
from grounded_vla.eval import EvalRunner
from pathlib import Path
import json

RUNS = Path('/kaggle/working/runs')

def run(agent_name, agent, ds_name, dataset, **kw):
    out = RUNS / f'{agent_name}__{ds_name}'
    runner = EvalRunner(agent)
    res = runner.evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True, **kw
    )
    print(f'{agent_name} on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

## ReAct + Mistral (Baseline 1)

Text-only. Quick on Mistral-7B with 4-bit quantization (~3–5 sec/task on T4).

In [ ]:
react = build_react()
_ = run('react_mistral', react, 'scienceqa',  scienceqa(limit=50))
_ = run('react_mistral', react, 'synthetic',  synthetic())
_ = run('react_mistral', react, 'mind2web',   mind2web(limit=30))
react.backend.close()

## Single-shot LLaVA (Baseline 2)

VLM, one call per task. Tests H1 (vision helps).

In [ ]:
ss = build_single_shot()
_ = run('single_shot_llava', ss, 'scienceqa',  scienceqa(limit=50))
_ = run('single_shot_llava', ss, 'synthetic',  synthetic())
_ = run('single_shot_llava', ss, 'mind2web',   mind2web(limit=30))
ss.backend.close()

## ORA + LLaVA (Our Method)

VLM with the Observe-Reason-Act loop and per-step visual re-encoding. Tests H2.

In [ ]:
ora = build_ora()
_ = run('ora_llava', ora, 'scienceqa',  scienceqa(limit=50))
_ = run('ora_llava', ora, 'synthetic',  synthetic())
_ = run('ora_llava', ora, 'mind2web',   mind2web(limit=30))
ora.backend.close()

In [ ]:
# Aggregate results table for the report.
import json
from pathlib import Path
rows = []
for d in sorted(Path('/kaggle/working/runs').iterdir()):
    s = json.loads((d / 'summary.json').read_text())
    rows.append((d.name, s['n_tasks'], s['task_completion_rate'], s['mean_steps'], s['error_breakdown']))
for r in rows:
    print(f'{r[0]:30s}  n={r[1]:3d}  success={r[2]:.3f}  steps={r[3]:.2f}  errors={r[4]}')

## Save results

Click **Save Version** → name the output dataset `gvla-runs-YYYY-MM-DD`. The next
notebook (LoRA + ablation) and your report code can mount it read-only.